<a href="https://colab.research.google.com/github/KasunUdayanga/NER-Sinhala-political-comment-identifier/blob/main/infoxml-base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Fine-tune a `microsoft/infoxlm-base` model for sequence labeling on the `sinhala_dataset_balanced.conll` dataset, including data loading, preprocessing, model configuration, training, and evaluation, and then summarize the model's performance and observations.

## Load and Inspect Dataset

### Subtask:
Load the `sinhala_dataset_balanced.conll` file and inspect its structure. This will involve understanding the CoNLL format, identifying sentences, and extracting words and their corresponding labels to prepare for tokenization and model input.


**Reasoning**:
I need to load the CoNLL file, parse its content, and store it in a structured format as per the instructions. This involves iterating through the file line by line to identify words, labels, and sentence boundaries.



In [1]:
sentences = []
current_sentence = []

with open('/content/sinhala_dataset_balanced.conll', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            # Assuming CoNLL format: word\tlabel or word label
            parts = line.split('\t')
            if len(parts) == 2:
                word, label = parts
                current_sentence.append((word, label))
            else:
                # Handle cases where separator might be space or other characters
                parts = line.split(' ')
                if len(parts) == 2:
                    word, label = parts
                    current_sentence.append((word, label))
                else:
                    # Skip malformed lines or log a warning if necessary
                    pass
        else:
            # Blank line indicates end of a sentence
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []

# Add the last sentence if the file doesn't end with a blank line
if current_sentence:
    sentences.append(current_sentence)


print(f"Total sentences parsed: {len(sentences)}")
print("\nFirst 3 sentences with words and labels:")
for i, sentence in enumerate(sentences[:3]):
    print(f"\n--- Sentence {i+1} ---")
    for word, label in sentence:
        print(f"Word: '{word}', Label: '{label}'")

Total sentences parsed: 9633

First 3 sentences with words and labels:

--- Sentence 1 ---
Word: 'පාලමුන', Label: 'B-LOC'

--- Sentence 2 ---
Word: 'ඔබතමයි', Label: 'B-Other'
Word: 'කාරයෝන්ට', Label: 'B-Other'
Word: 'රාජපක්ෂලාටම්', Label: 'B-PER'
Word: 'හර්ශ', Label: 'B-PER'
Word: 'විපක්ශයට', Label: 'B-ORG'
Word: 'අවලද', Label: 'B-Other'
Word: 'දැම්මාඇත්ත', Label: 'B-Other'
Word: 'ඩොබිලාට', Label: 'B-Other'
Word: 'පැන්චො', Label: 'B-Other'

--- Sentence 3 ---
Word: 'මකුලෑව', Label: 'B-LOC'


## Load Pre-trained Model and Tokenizer

### Subtask:
Load the `microsoft/infoxlm-base` pre-trained model and its associated tokenizer using the `transformers` library. This step ensures we have the correct vocabulary and model architecture for fine-tuning.


**Reasoning**:
Before importing and using the `transformers` library, I need to ensure it is installed in the environment. This is a prerequisite for loading the tokenizer and model.



In [2]:
pip install transformers

**Reasoning**:
Now that the `transformers` library is confirmed to be installed, I will proceed with loading the tokenizer and model as specified in the subtask. This involves importing necessary classes, defining the model name, extracting unique labels from the previously processed `sentences` data, creating label-to-ID and ID-to-label mappings, and then loading the pre-trained model for token classification.



In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "microsoft/infoxlm-base"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer '{model_name}' loaded successfully.")

# Extract unique labels from the sentences data
all_labels = sorted(list(set(label for sentence in sentences for _, label in sentence)))

# Create label-to-id and id-to-label mappings
label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for i, label in enumerate(all_labels)}
num_labels = len(all_labels)

print(f"\nFound {num_labels} unique labels: {all_labels}")
print(f"Label to ID mapping: {label_to_id}")
print(f"ID to Label mapping: {id_to_label}")

# Load the pre-trained model for token classification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)
print(f"\nModel '{model_name}' for token classification loaded successfully with {num_labels} labels.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer 'microsoft/infoxlm-base' loaded successfully.

Found 9 unique labels: ['B-LOC', 'B-ORG', 'B-Other', 'B-PER', 'I-LOC', 'I-ORG', 'I-Other', 'I-PER', 'O']
Label to ID mapping: {'B-LOC': 0, 'B-ORG': 1, 'B-Other': 2, 'B-PER': 3, 'I-LOC': 4, 'I-ORG': 5, 'I-Other': 6, 'I-PER': 7, 'O': 8}
ID to Label mapping: {0: 'B-LOC', 1: 'B-ORG', 2: 'B-Other', 3: 'B-PER', 4: 'I-LOC', 5: 'I-ORG', 6: 'I-Other', 7: 'I-PER', 8: 'O'}


pytorch_model.bin:   0%|          | 0.00/942M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/942M [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at microsoft/infoxlm-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model 'microsoft/infoxlm-base' for token classification loaded successfully with 9 labels.


## Preprocess Dataset for Fine-tuning

### Subtask:
Tokenize the words from the dataset using the loaded tokenizer. Crucially, align the labels from the CoNLL file with the tokenized inputs, handling subword tokens correctly. This will create input IDs, attention masks, and aligned labels suitable for model training.


**Reasoning**:
The previous code block contained markdown in a code cell, leading to a `SyntaxError`. To correctly display the subtask description and proceed with the data preprocessing, I will first correct the previous cell to be a markdown cell and then create a new code cell to implement the label alignment and tokenization logic as per the instructions.



In [20]:
input_ids = []
attention_masks = []
aligned_labels = []

for sentence in sentences:
    words = [word for word, label in sentence]
    labels = [label for word, label in sentence]

    # Tokenize words and get word IDs
    # is_split_into_words=True indicates that the input is a list of words
    tokenized_input = tokenizer(words, is_split_into_words=True, truncation=True, padding='max_length', max_length=128)

    word_ids = tokenized_input.word_ids(batch_index=0)

    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        # Special tokens have a word ID of None. We set the label to -100 so they are ignored in the loss function.
        if word_idx is None:
            label_ids.append(-100)
        # We set the label for the first token of each word.
        elif word_idx != previous_word_idx:
            label_ids.append(label_to_id[labels[word_idx]])
        # For other tokens in a word, we set the label to -100.
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx

    input_ids.append(tokenized_input['input_ids'])
    attention_masks.append(tokenized_input['attention_mask'])
    aligned_labels.append(label_ids)

print(f"Processed {len(input_ids)} sentences.")
print(f"First sentence input IDs: {input_ids[0]}")
print(f"First sentence attention mask: {attention_masks[0]}")
print(f"First sentence aligned labels: {aligned_labels[0]}")

# Decode first tokenized sentence for verification
decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print(f"\nDecoded tokens for first sentence: {decoded_tokens}")
print(f"Corresponding aligned labels: {[id_to_label[l] if l != -100 else 'IGNORE' for l in aligned_labels[0]]}")

Processed 9633 sentences.
First sentence input IDs: [0, 237394, 80078, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
First sentence attention mask: [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
First sentence aligned labels: [-100, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -

**Reasoning**:
The previous step successfully processed the raw sentences into tokenized inputs and aligned labels. To prepare this data for model training, I need to convert these lists into PyTorch tensors and then create a PyTorch Dataset object. This will allow for efficient batching and feeding of data to the model during training.



In [21]:
import torch
from torch.utils.data import TensorDataset, random_split

# Convert lists to PyTorch tensors
input_ids_tensor = torch.tensor(input_ids)
attention_masks_tensor = torch.tensor(attention_masks)
aligned_labels_tensor = torch.tensor(aligned_labels)

# Create a TensorDataset
dataset = TensorDataset(input_ids_tensor, attention_masks_tensor, aligned_labels_tensor)

print(f"Total dataset size: {len(dataset)}")
print(f"Example input_ids tensor shape: {input_ids_tensor[0].shape}")
print(f"Example attention_mask tensor shape: {attention_masks_tensor[0].shape}")
print(f"Example aligned_labels tensor shape: {aligned_labels_tensor[0].shape}")

# Define the proportion for training and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Split the dataset into training and validation sets
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Total dataset size: 9633
Example input_ids tensor shape: torch.Size([128])
Example attention_mask tensor shape: torch.Size([128])
Example aligned_labels tensor shape: torch.Size([128])
Training dataset size: 7706
Validation dataset size: 1927


## Configure Training and Fine-tune Model

### Subtask:
Define the training arguments, including parameters like batch size, learning rate, and number of epochs. Then, fine-tune the `microsoft/infoxlm-base` model on the preprocessed dataset using a `Trainer` from the `transformers` library.


In [27]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from transformers.trainer_utils import IntervalStrategy
from torch.utils.data import Dataset as TorchDataset # Import torch's Dataset to avoid name conflict

# Define a custom Dataset class to return dictionaries
class DictOutputDataset(TorchDataset):
    def __init__(self, subset_dataset):
        self.subset_dataset = subset_dataset

    def __len__(self):
        return len(self.subset_dataset)

    def __getitem__(self, idx):
        input_ids, attention_mask, labels = self.subset_dataset[idx]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy=IntervalStrategy.EPOCH,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy=IntervalStrategy.EPOCH,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

# Initialize Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Wrap the existing train_dataset and val_dataset with DictOutputDataset
train_dataset_dict = DictOutputDataset(train_dataset)
val_dataset_dict = DictOutputDataset(val_dataset)

# Create a Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_dict,
    eval_dataset=val_dataset_dict,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

print("Model training initiated.")


/tmp/ipython-input-780169153.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,1.035600,0.962471
2,0.789900,0.782541
3,0.810000,0.747492


Model training initiated.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from transformers.trainer_utils import IntervalStrategy
from torch.utils.data import Dataset, random_split

# --- 1. Load and Inspect Dataset (from cell 13c8a621) ---
sentences = []
current_sentence = []

with open('/content/sinhala_dataset_balanced.conll', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split('\t')
            if len(parts) == 2:
                word, label = parts
                current_sentence.append((word, label))
            else:
                parts = line.split(' ')
                if len(parts) == 2:
                    word, label = parts
                    current_sentence.append((word, label))
        else:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []

if current_sentence:
    sentences.append(current_sentence)

# --- 2. Load Pre-trained Model and Tokenizer (from cell 8bb17d2d) ---
model_name = "microsoft/infoxlm-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

all_labels = sorted(list(set(label for sentence in sentences for _, label in sentence)))
label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for i, label in enumerate(all_labels)}
num_labels = len(all_labels)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)

# --- 3. Preprocess Data for Tokenization (from cell bf73b392) ---
input_ids = []
attention_masks = []
aligned_labels = []

for sentence in sentences:
    words = [word for word, label in sentence]
    labels = [label for word, label in sentence]

    tokenized_input = tokenizer(words, is_split_into_words=True, truncation=True, padding='max_length', max_length=128)

    word_ids = tokenized_input.word_ids(batch_index=0)

    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label_to_id[labels[word_idx]])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx

    input_ids.append(tokenized_input['input_ids'])
    attention_masks.append(tokenized_input['attention_mask'])
    aligned_labels.append(label_ids)

# --- 4. Prepare Dataset for PyTorch (from fca8a1de, modified for DictOutputDataset)---
# Define a custom Dataset class that returns dictionaries directly
class NamedTensorDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "labels": self.labels[idx]
        }

# Convert lists to PyTorch tensors
input_ids_tensor = torch.tensor(input_ids)
attention_masks_tensor = torch.tensor(attention_masks)
aligned_labels_tensor = torch.tensor(aligned_labels)

# Create a NamedTensorDataset
full_dataset = NamedTensorDataset(input_ids_tensor, attention_masks_tensor, aligned_labels_tensor)

# Define the proportion for training and validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# Split the dataset into training and validation sets
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")


# --- 5. Configure Training and Fine-tune Model ---
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy=IntervalStrategy.EPOCH,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy=IntervalStrategy.EPOCH,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

# Initialize Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Create a Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()

print("Model training initiated.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/942M [00:00<?, ?B/s]

...

/tmp/ipython-input-1980379121.py:134: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kalihunter418 (kalihunter418-university-of-vauniya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,1.016000,0.848980
2,0.786100,0.730337
3,0.759900,0.711138


Model training initiated.


## Save Fine-tuned Model to Google Drive

**Reasoning**:
To save the trained model to Google Drive, I first need to mount the drive to access its file system. After that, I'll define a suitable path within the drive and use the `trainer.save_model()` method, which conveniently saves both the model's weights and the tokenizer's configuration, ensuring that the entire setup can be reloaded later.

In [11]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define a path in your Google Drive to save the model
model_save_path = "/content/drive/My Drive/infoxlm-base-finetuned-sinhala"

# Save the fine-tuned model and tokenizer
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to: {model_save_path}")

Mounted at /content/drive
Model and tokenizer saved to: /content/drive/My Drive/infoxlm-base-finetuned-sinhala


## Evaluate Model Performance

### Subtask:
Evaluate the fine-tuned model on a separate test set (if available, otherwise a portion of the training data) using relevant metrics for the specific NLP task (e.g., precision, recall, F1-score for sequence labeling, or accuracy for classification). Plotting the evaluation results might be included as part of this step.


**Reasoning**:
Now that the model has been fine-tuned, it's crucial to evaluate its performance using appropriate metrics. I will use the `Trainer.evaluate()` method to get the loss and relevant metrics (precision, recall, f1-score) on the validation set. I'll need to define a `compute_metrics` function for the Trainer to use, which will calculate these metrics based on the predicted and true labels.


In [8]:
!pip install evaluate
!pip install seqeval

import numpy as np
import evaluate # Changed from from datasets import load_metric


# Load seqeval metric
metric = evaluate.load("seqeval") # Changed from load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[id_to_label[l] for l in label if l != -100] for label in labels]
    true_predictions = [[id_to_label[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Re-initialize Trainer with compute_metrics
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Evaluate the model
eval_results = trainer.evaluate()

print(f"Evaluation Results: {eval_results}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6610e59040b46050b13c58a6b2d29ff44ba93162060545c35b2be6ea49db20e9
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


/tmp/ipython-input-4115233185.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation Results: {'eval_loss': 0.7111382484436035, 'eval_model_preparation_time': 0.0045, 'eval_precision': 0.7094614967135018, 'eval_recall': 0.9016707541385653, 'eval_f1': 0.7941007728392562, 'eval_accuracy': 0.7124100201554852, 'eval_runtime': 15.0033, 'eval_samples_per_second': 128.439, 'eval_steps_per_second': 8.065}


# Task
The fine-tuning process involved several key stages. First, the `sinhala_dataset_balanced.conll` dataset was loaded and parsed to extract words and their corresponding labels. We identified 9 unique labels, including 'B-LOC', 'B-ORG', 'B-Other', 'B-PER', 'I-LOC', 'I-ORG', 'I-Other', 'I-PER', and 'O'.

Next, the `microsoft/infoxlm-base` pre-trained model and its tokenizer were loaded. The dataset was then preprocessed, tokenizing each sentence and aligning labels to subword tokens, with special tokens assigned a label of -100 to be ignored during loss calculation. This resulted in a dataset of 9633 sentences, which was split into a training set of 7706 samples and a validation set of 1927 samples.

The model was fine-tuned for 3 epochs using the `transformers` `Trainer` API, with a learning rate of 2e-5 and a batch size of 16. The training process leveraged `DataCollatorForTokenClassification` and a custom `NamedTensorDataset` to ensure correct input formatting.

Upon evaluation, the model achieved the following performance metrics on the validation set:
- **Loss:** 0.711
- **Precision:** 0.709
- **Recall:** 0.902
- **F1-score:** 0.794
- **Accuracy:** 0.712

**Key Observations:**
The model demonstrates a strong recall score (0.902), suggesting it is effective at identifying most of the named entities in the text. However, the precision (0.709) and F1-score (0.794) indicate there's room for improvement in correctly classifying these entities without false positives. The overall accuracy of 0.712 is reasonable but highlights the complexity of the task. The model might be slightly over-predicting entities, leading to higher recall but lower precision.

**Potential Next Steps:**
1.  **Hyperparameter Tuning:** Experiment with different learning rates, batch sizes, and number of training epochs to optimize performance.
2.  **Advanced Tokenization and Label Alignment:** Investigate more sophisticated label alignment strategies, especially for subword tokens, to see if it improves precision.
3.  **Data Augmentation:** Expand the training dataset with more diverse examples to improve the model's generalization capabilities.
4.  **Error Analysis:** Analyze the specific types of errors the model is making (e.g., confusion between 'B-Other' and other entity types, or misclassifying 'I-' tags) to identify areas for targeted improvement.
5.  **Different Architectures:** Explore other transformer models that might be better suited for Sinhala language processing or token classification tasks.
6.  **Performance Visualization:** Plot training and validation loss, and metric scores over epochs to better understand the training dynamics and identify potential overfitting.

## Final Task

### Subtask:
Summarize the entire fine-tuning process, highlighting the performance of the model on your dataset and discussing any key observations or potential next steps.


## Summary:

### Q&A
The fine-tuning process involved loading and parsing the `sinhala_dataset_balanced.conll` dataset, which contained 9 unique named entity labels. The `microsoft/infoxlm-base` pre-trained model was used, and the dataset was preprocessed by tokenizing sentences and aligning labels to subword tokens. This resulted in a training set of 7706 samples and a validation set of 1927 samples. The model was fine-tuned for 3 epochs with a learning rate of 2e-5 and a batch size of 16 using the `transformers` `Trainer` API.

The model achieved a precision of 0.709, recall of 0.902, F1-score of 0.794, and an accuracy of 0.712 on the validation set. Key observations include a strong recall score indicating effectiveness in identifying named entities, but a lower precision suggesting potential over-prediction of entities or false positives.

Potential next steps include hyperparameter tuning, exploring advanced tokenization and label alignment strategies, and conducting error analysis to pinpoint specific areas for improvement.

### Data Analysis Key Findings
*   The fine-tuned `microsoft/infoxlm-base` model achieved a recall of 0.902 on the validation set, indicating its strong ability to identify named entities.
*   The model's precision was 0.709, and its F1-score was 0.794, suggesting there is room for improvement in correctly classifying identified entities without generating false positives.
*   The overall accuracy of the model on the token classification task was 0.712, with a loss of 0.711.
*   The dataset contained 9 unique named entity labels and was split into 7706 training samples and 1927 validation samples.

### Insights or Next Steps
*   Focus on improving precision through methods like advanced label alignment or error analysis, as the current high recall suggests the model might be over-predicting entities.
*   Conduct hyperparameter tuning, experiment with data augmentation, or explore different model architectures to further optimize performance and generalization.
